In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [6]:
### load dataset 

data=pd.read_excel('Churn_Modelling.xlsx')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
data = data.drop(columns=["RowNumber", "CustomerId", "Surname"], errors="ignore")
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [12]:
## encode one categorical variable
label_endcoder_gender = LabelEncoder()
data["Gender"] = label_endcoder_gender.fit_transform(data["Gender"])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
#onehot for geography
from sklearn.preprocessing import OneHotEncode
onehotencoder = OneHotEncoder()
geo_encoder = onehotencoder.fit_transform(data[["Geography"]])

In [18]:
geo_df = pd.DataFrame(geo_encoder.toarray(), columns=onehotencoder.get_feature_names_out(['Geography']))
geo_df.head(100)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
95,0.0,0.0,1.0
96,0.0,0.0,1.0
97,0.0,1.0,0.0
98,0.0,0.0,1.0


In [20]:
#combine the onehot encoded columns with the original dataframe

data = pd.concat([data.drop(columns=['Geography']), geo_df], axis=1)

In [22]:
with open("label_encoder_geo.pkl", "wb") as f:
    pickle.dump(label_endcoder_gender, f)

In [23]:
with open("onehot_encoder_geo.pkl", "wb") as f:
    pickle.dump(onehotencoder, f)

In [26]:
X = data.drop(columns=["Exited"])   #not depending variable
y = data["Exited"]   #depending variable
#### traning and testing data split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

####Scale this features
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler,f)
    

In [28]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [33]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [35]:
(X_train.shape[1],)

(12,)

In [37]:
## build model 
model =Sequential([
       Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
       Dense(16, activation="relu"),
       Dense(1, activation="sigmoid")
])

In [38]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                416       
                                                                 
 dense_1 (Dense)             (None, 16)                528       
                                                                 
 dense_2 (Dense)             (None, 1)                 17        
                                                                 
Total params: 961 (3.75 KB)
Trainable params: 961 (3.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [40]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
tensorflow.keras.losses.BinaryCrossentropy()

In [41]:
model.compile(optimizer=(opt),loss="binary_crossentropy",metrics=["accuracy"])

In [47]:
## setup tensorboard
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [49]:
### early stopping 
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [50]:
history = model.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test), callbacks=[tensorboard_callback, early_stopping])

Epoch 1/100


250/250 [==============================] - 3s 4ms/step - loss: 0.4207 - accuracy: 0.8165 - val_loss: 0.3609 - val_accuracy: 0.8485
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3562 - accuracy: 0.8524 - val_loss: 0.3560 - val_accuracy: 0.8515
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3482 - accuracy: 0.8568 - val_loss: 0.3567 - val_accuracy: 0.8570
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3438 - accuracy: 0.8564 - val_loss: 0.3382 - val_accuracy: 0.8630
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3402 - accuracy: 0.8631 - val_loss: 0.3346 - val_accuracy: 0.8650
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3386 - accuracy: 0.8605 - val_loss: 0.3336 - val_accuracy: 0.8625
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3362 - accuracy: 0.8610 - val_loss: 0.3364 - val_accuracy: 0.86

In [51]:
model.save("model.h5")

c:\Users\rajku\ANN\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [56]:
%load_ext tensorboard
%tensorboard --logdir logs/fit20260617-171308

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 20492), started 0:00:17 ago. (Use '!kill 20492' to kill it.)